# Sentinel-1 SAR workflow for tree-covered coffee detectability

This notebook is part of the reproducibility material associated with the manuscript:

**Detectability limits of tree-covered coffee at Sentinel resolution: asymmetric gains from optical texture**

## Purpose

This notebook extracts and evaluates a parsimonious Sentinel-1 SAR feature set for distinguishing tree-covered coffee from forest formations in the Dominican Republic. The workflow is designed to document the SAR-only component of the study and to generate sample-level attributes that can be used independently or integrated with Sentinel-2 optical and texture predictors.

## Scientific rationale

The Sentinel-1 baseline is restricted to variables with direct structural or acquisition-related interpretation:

1. `VV_med` — median co-polarised backscatter level;
2. `VH_med` — median cross-polarised, volume-related backscatter level;
3. `ratio_med` — median VH/VV structural relation in linear scale;
4. `ratio_sd` — temporal variability of the VH/VV structural relation;
5. `ratio_p90_minus_p10` — robust temporal amplitude of the VH/VV structural relation.

`VV_count` and `angle_med` are retained as quality-control or diagnostic fields, but they are not used as predictors in the main SAR Random Forest model.

## Data privacy and reproducibility note

The notebook expects a local polygon dataset containing the reference samples. The original polygon geometries are not included in the public repository because they may reveal sensitive field locations or institutional reference data. Users wishing to reproduce the workflow should prepare their own vector dataset with the required attributes described below.

Minimum expected attributes:

- `Class`: target class, with values such as `coffee` and `forest`;
- `New_ID`: optional coffee shade-level code, where `1` indicates low shade / coffee visible, `2` indicates intermediate shade, and `3` indicates high shade / forest-like coffee.

A stable `sample_id` is generated internally from the row order after reading the vector file.


## Notebook organisation

The workflow is organised into the following blocks:

1. Environment configuration and Google Earth Engine initialisation;
2. Reading and standardising the reference polygon dataset;
3. Sentinel-1 acquisition for the DJFM seasonal windows;
4. SAR pre-processing and feature construction;
5. Polygon-based extraction and export of SAR attributes;
6. Optional SAR texture diagnostics;
7. Univariate separability analysis;
8. Random Forest SAR-only modelling;
9. Error analysis and optional figure export;
10. Reproducibility summary and AI-use statement.

All code cells are intended to be run sequentially. The optional GLCM texture blocks can be skipped if only the baseline SAR feature set is required.


## Block 1 — Environment configuration and Google Earth Engine initialisation

This block installs and imports the Python packages required for the workflow, authenticates Google Earth Engine and defines user-specific paths.

Before running the notebook, update the configuration variables in the code cell below. Personal paths, cloud project identifiers and private filenames should not be committed to a public repository.


In [ ]:
# ============================================================
# Environment configuration
# ============================================================

# Install dependencies when running in Google Colab.
# In a local environment, these packages can also be installed from requirements.txt.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install earthengine-api geemap geopandas

# Core imports
import os
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: mount Google Drive when running in Colab.
# Set to False if the input vector file is stored elsewhere.
USE_GOOGLE_DRIVE = True

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# ============================================================
# User configuration
# ============================================================

# Google Earth Engine project.
# For public repositories, avoid hard-coding personal or institutional project IDs.
# Use None for default initialisation, or set your project ID locally before running.
GEE_PROJECT_ID = None  # Example for local use only: "your-gee-project-id"

# Path to the local/private reference polygon dataset.
# This file is intentionally not included in the public GitHub repository.
INPUT_VECTOR_PATH = "/content/drive/MyDrive/path/to/reference_polygons.shp"

# Google Drive folder used for Earth Engine CSV exports.
DRIVE_EXPORT_FOLDER = "sentinel_coffee_outputs"

# Local/Drive folder for optional table exports.
TABLE_OUTPUT_DIR = "/content/drive/MyDrive/sentinel_coffee_outputs/tables"

# Local/Drive folder for optional figure exports.
FIGURE_OUTPUT_DIR = "/content/drive/MyDrive/sentinel_coffee_outputs/figures"

# ============================================================
# Earth Engine authentication and initialisation
# ============================================================

ee.Authenticate()

if GEE_PROJECT_ID is None:
    ee.Initialize()
else:
    ee.Initialize(project=GEE_PROJECT_ID)

print("Environment configured successfully.")


## Block 2 — Reference polygons and coffee shade groups

This block reads the reference polygon dataset, standardises the class labels and creates the grouping variable used to analyse coffee under different visual shade conditions.

The geometry is used only to extract satellite attributes. For public sharing, the polygon file itself should remain outside the repository unless it has been explicitly authorised for redistribution and does not expose sensitive field locations.


In [ ]:
# ============================================================
# Read and standardise the reference polygon dataset
# ============================================================

if not os.path.exists(INPUT_VECTOR_PATH):
    raise FileNotFoundError(
        "The reference polygon file was not found. "
        "Update INPUT_VECTOR_PATH in Block 1 before running this notebook."
    )

gdf = gpd.read_file(INPUT_VECTOR_PATH)

# Ensure CRS is EPSG:4326 for compatibility with Earth Engine.
if gdf.crs is None:
    print("CRS not defined. Setting as EPSG:4326 (assumed).")
    gdf = gdf.set_crs("EPSG:4326")

if str(gdf.crs).upper() != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# Standardise class field.
required_input_columns = ["Class", "New_ID"]
missing_input_columns = [col for col in required_input_columns if col not in gdf.columns]
if missing_input_columns:
    raise ValueError(
        "Missing required input columns: "
        f"{missing_input_columns}. Expected at least: {required_input_columns}"
    )

gdf["Class"] = gdf["Class"].astype(str).str.strip().str.lower()

# New_ID interpretation:
# 1 = low shade / coffee visible
# 2 = intermediate shade
# 3 = high shade / forest-like
gdf["New_ID"] = pd.to_numeric(gdf["New_ID"], errors="coerce")
gdf.loc[gdf["Class"] == "forest", "New_ID"] = np.nan

# Stable sample ID generated from the input row order.
gdf = gdf.reset_index(drop=True)
gdf["sample_id"] = ["id_%03d" % i for i in range(len(gdf))]

def assign_group(row):
    if row["Class"] == "forest":
        return "forest"
    if row["Class"] == "coffee" and row["New_ID"] == 1:
        return "coffee_low_shade"
    if row["Class"] == "coffee" and row["New_ID"] == 2:
        return "coffee_intermediate_shade"
    if row["Class"] == "coffee" and row["New_ID"] == 3:
        return "coffee_high_shade"
    return "unknown"

gdf["group"] = gdf.apply(assign_group, axis=1)

print("Polygons loaded:", gdf.shape)
print("\nClass distribution:")
print(gdf["Class"].value_counts())
print("\nCoffee shade-level distribution:")
print(gdf.loc[gdf["Class"] == "coffee", "New_ID"].value_counts(dropna=False).sort_index())
print("\nGroup distribution:")
print(gdf["group"].value_counts(dropna=False))

def gdf_to_fc(gdf):
    features = []
    for _, row in gdf.iterrows():
        geom = ee.Geometry(row.geometry.__geo_interface__)
        new_id = row["New_ID"]

        if pd.isna(new_id):
            new_id = -9999
        else:
            new_id = int(new_id)

        features.append(
            ee.Feature(
                geom,
                {
                    "class": row["Class"],
                    "sample_id": row["sample_id"],
                    "New_ID": new_id,
                    "shade_level": new_id,
                    "group": row["group"],
                },
            )
        )

    return ee.FeatureCollection(features)

samples_fc = gdf_to_fc(gdf)
coffee_fc = samples_fc.filter(ee.Filter.eq("class", "coffee"))
forest_fc = samples_fc.filter(ee.Filter.eq("class", "forest"))

print("\nTotal sample polygons:", samples_fc.size().getInfo())
print("Coffee polygons:", coffee_fc.size().getInfo())
print("Forest polygons:", forest_fc.size().getInfo())

# ROI from sample extent with 3 km buffer.
roi = samples_fc.geometry().bounds().buffer(3000)
print("\nROI defined from sample extent with 3 km buffer.")


## Block 3 — Sentinel-1 acquisition: DJFM seasonal window

This block filters Sentinel-1 GRD imagery over the reference-sample region. The seasonal window focuses on December–March (DJFM), which matches the period used for the multitemporal analysis in the manuscript.

Ascending and descending orbits are reported separately for quality control, but the SAR feature construction below uses the merged collection.


In [ ]:
# DJFM windows used for Sentinel-1, consistent with the S2 seasonal design
SEASONS = [
    ("2022-12-01", "2023-03-31"),
    ("2023-12-01", "2024-03-31"),
    ("2024-12-01", "2025-03-31"),
]

def get_s1_season(start_date, end_date, roi_geom):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi_geom)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH", "angle"])
    )

s1_cols = [get_s1_season(start, end, roi) for start, end in SEASONS]
s1 = s1_cols[0].merge(s1_cols[1]).merge(s1_cols[2])

s1_asc = s1.filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
s1_desc = s1.filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))

print("Total S1 images:", s1.size().getInfo())
print("ASC images:", s1_asc.size().getInfo())
print("DESC images:", s1_desc.size().getInfo())

def first_dates(col, n=8):
    ms = (
        col.sort("system:time_start")
        .limit(n)
        .aggregate_array("system:time_start")
        .getInfo()
    )
    return [ee.Date(t).format("YYYY-MM-dd").getInfo() for t in ms]

print("First S1 dates:", first_dates(s1, 8))

## Block 4 — Sentinel-1 pre-processing

This block converts Sentinel-1 VV and VH backscatter from dB to linear scale and applies a broad dB-range mask to remove extreme or invalid values.

The VH/VV ratio is calculated in linear scale because ratios between backscatter intensities are physically more meaningful before logarithmic transformation.


In [ ]:
def add_db_and_linear(img):
    # Sentinel-1 GRD backscatter in GEE is in dB
    vv_db = img.select("VV").rename("VV_db")
    vh_db = img.select("VH").rename("VH_db")
    angle = img.select("angle").rename("angle")

    # Convert dB to linear scale for physically meaningful ratios
    vv_lin = ee.Image.constant(10).pow(vv_db.divide(10)).rename("VV_lin")
    vh_lin = ee.Image.constant(10).pow(vh_db.divide(10)).rename("VH_lin")

    out = ee.Image.cat([vv_db, vh_db, vv_lin, vh_lin, angle])
    return out.copyProperties(img, img.propertyNames())

def mask_basic_db(img, db_min=-40, db_max=5):
    vv_db = img.select("VV_db")
    vh_db = img.select("VH_db")

    mask = (
        vv_db.gte(db_min).And(vv_db.lte(db_max))
        .And(vh_db.gte(db_min).And(vh_db.lte(db_max)))
    )

    return img.updateMask(mask)

s1_prep = (
    s1
    .map(add_db_and_linear)
    .map(mask_basic_db)
)

print("Preprocessed S1 images:", s1_prep.size().getInfo())

# QA: median dB min/max over ROI
med_db = s1_prep.select(["VV_db", "VH_db"]).median().clip(roi)

qa_minmax = med_db.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=1000,
    bestEffort=True,
    maxPixels=1e9
).getInfo()

print("Median dB min/max over ROI:")
print(qa_minmax)

## Block 5 — SAR feature construction

This block creates the SAR predictors used in the baseline model. The feature set emphasises central tendency and temporal variability of the VH/VV structural relation, which is expected to respond to canopy structure, volume scattering and vegetation complexity.

The count and incidence-angle layers are exported for diagnostics, but they are not used as predictors in the main SAR-only Random Forest model.


In [ ]:
def build_s1_features_main(col, prefix):
    '''
    Main analytical SAR features:
    - VV median in dB
    - VH median in dB
    - VH/VV median in linear scale
    - VH/VV standard deviation in linear scale
    - VH/VV robust range: p90 - p10 in linear scale

    QA/diagnostic fields:
    - VV_count
    - angle_med
    '''

    vv_db = col.select("VV_db")
    vh_db = col.select("VH_db")

    # Per-image VH/VV ratio in linear scale
    ratio_col = col.map(
        lambda img: img.addBands(
            img.select("VH_lin")
               .divide(img.select("VV_lin"))
               .rename("VH_over_VV_lin")
        )
    ).select("VH_over_VV_lin")

    # Core central tendency
    vv_med = vv_db.median().rename(f"{prefix}_VV_med")
    vh_med = vh_db.median().rename(f"{prefix}_VH_med")
    ratio_med = ratio_col.median().rename(f"{prefix}_ratio_med")

    # Temporal variability and robust amplitude of VH/VV
    ratio_sd = ratio_col.reduce(ee.Reducer.stdDev()).rename(f"{prefix}_ratio_sd")

    ratio_pct = ratio_col.reduce(ee.Reducer.percentile([10, 90]))
    ratio_p10 = ratio_pct.select("VH_over_VV_lin_p10").rename(f"{prefix}_ratio_p10")
    ratio_p90 = ratio_pct.select("VH_over_VV_lin_p90").rename(f"{prefix}_ratio_p90")
    ratio_p90_minus_p10 = (
        ratio_p90.subtract(ratio_p10)
        .rename(f"{prefix}_ratio_p90_minus_p10")
    )

    # QA / diagnostic fields, not used in the main model
    vv_count = vv_db.reduce(ee.Reducer.count()).rename(f"{prefix}_VV_count")
    angle_med = col.select("angle").median().rename(f"{prefix}_angle_med")

    return ee.Image.cat([
        vv_med,
        vh_med,
        ratio_med,
        ratio_sd,
        ratio_p90_minus_p10,
        vv_count,
        angle_med
    ])

s1_feat_total = build_s1_features_main(s1_prep, "DJFM_22_25").clip(roi)

print("SAR feature band count:", s1_feat_total.bandNames().size().getInfo())
print("SAR bands:")
print(s1_feat_total.bandNames().getInfo())

## Block 6 — Polygon-based extraction and validity filtering

This block extracts SAR attributes for each reference polygon using mean aggregation. A minimum valid-image count is applied to reduce the influence of polygons with insufficient temporal coverage.

The resulting table is the main SAR attribute table used in the separability analysis and SAR-only classification model.


In [ ]:
# Extract SAR features over sample polygons
samples_out = s1_feat_total.reduceRegions(
    collection=samples_fc,
    reducer=ee.Reducer.mean(),
    scale=10,
    tileScale=4
)

# Validity filters
valid_field = "DJFM_22_25_VV_med"
count_field = "DJFM_22_25_VV_count"
min_count = 15

valid = samples_out.filter(ee.Filter.neq(valid_field, None))
valid2 = valid.filter(ee.Filter.gte(count_field, min_count))

print("Valid polygons (non-null VV_med):", valid.size().getInfo())
print(f"Valid polygons with {count_field} >= {min_count}:", valid2.size().getInfo())

print("\nClass counts after filtering:")
print(valid2.aggregate_histogram("class").getInfo())

print("\nGroup counts after filtering:")
print(valid2.aggregate_histogram("group").getInfo())

# Convert to DataFrame
sar_df = geemap.ee_to_df(valid2)

# Standardise fields
sar_df["class"] = sar_df["class"].astype(str).str.strip().str.lower()
sar_df["group"] = sar_df["group"].astype(str).str.strip().str.lower()

sar_df["New_ID"] = pd.to_numeric(sar_df["New_ID"], errors="coerce")
sar_df["shade_level"] = pd.to_numeric(sar_df["shade_level"], errors="coerce")

# Convert Earth Engine missing code back to NaN
sar_df.loc[sar_df["New_ID"] == -9999, "New_ID"] = np.nan
sar_df.loc[sar_df["shade_level"] == -9999, "shade_level"] = np.nan

# Forest polygons do not have coffee shade level
sar_df.loc[sar_df["class"] == "forest", "New_ID"] = np.nan
sar_df.loc[sar_df["class"] == "forest", "shade_level"] = np.nan

print("\nRows:", sar_df.shape[0])
print("Columns:", sar_df.shape[1])

print("\nClass distribution:")
print(sar_df["class"].value_counts(dropna=False))

print("\nCoffee shade-level distribution:")
print(
    sar_df.loc[sar_df["class"] == "coffee", "shade_level"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nGroup distribution:")
print(sar_df["group"].value_counts(dropna=False))

display(sar_df.head())
display(sar_df.isna().sum().sort_values(ascending=False).head(20))

## Block 7 — Export SAR attribute table

This block exports the polygon-level SAR attribute table to Google Drive. The export includes the analytical SAR features and quality-control fields.

The exported CSV can be used as an intermediate table for subsequent modelling or integration with Sentinel-2 optical predictors.


In [ ]:
# Export full SAR table, including analytical features and QA fields.
task_sar = ee.batch.Export.table.toDrive(
    collection=valid2,
    description="SAR_features_DJFM_22_25_main_noDEM",
    folder=DRIVE_EXPORT_FOLDER,
    fileNamePrefix="SAR_features_DJFM_22_25_main_noDEM",
    fileFormat="CSV",
)

task_sar.start()
print("SAR export started: SAR_features_DJFM_22_25_main_noDEM.csv")


## Block 7B — Optional Sentinel-1 GLCM texture from VH/VV ratio

This block computes a deliberately restricted SAR texture set from `DJFM_22_25_ratio_med` only. The purpose is to test whether spatial texture in the structural SAR ratio adds information beyond the baseline SAR variables. The variables generated here are optional and should be evaluated before being included in the final integration model.


In [ ]:
# ============================================================
# OPTIONAL BLOCK — SENTINEL-1 GLCM TEXTURE
# Texture derived only from DJFM_22_25_ratio_med
# ============================================================

print("\n" + "="*70)
print("SENTINEL-1 GLCM TEXTURE — OPTIONAL BLOCK")
print("="*70)

# Scientific rationale:
# The VH/VV ratio family was the most informative SAR component.
# Therefore, SAR texture is computed only from DJFM_22_25_ratio_med.

target_band = "DJFM_22_25_ratio_med"

# Use the SAR feature image created in Block 5.
# If the notebook was run from the start, s1_feat_total must exist here.
if "s1_feat_total" not in globals():
    raise NameError(
        "s1_feat_total is not defined. Run Blocks 1–5 before the S1 GLCM block."
    )

available_bands = s1_feat_total.bandNames().getInfo()
print("\nAvailable SAR feature bands:")
print(available_bands)

if target_band not in available_bands:
    raise ValueError(
        f"Required band '{target_band}' was not found in s1_feat_total."
    )

ratio_glcm_base = s1_feat_total.select(target_band).rename("ratio")

print("\nBase image for S1 GLCM:")
print(target_band)

# ------------------------------------------------------------
# Quantisation for GLCM
# ------------------------------------------------------------
# GLCM requires integer input.
# The VH/VV ratio is scaled to byte values using a broad vegetation range.
# Values outside the range are clipped to reduce the effect of outliers.

ratio_glcm_uint8 = (
    ratio_glcm_base
    .unitScale(0.05, 0.60)
    .clamp(0, 1)
    .multiply(255)
    .toByte()
)

# ------------------------------------------------------------
# Compute GLCM texture
# ------------------------------------------------------------
# size=3 corresponds to a 3 × 3 moving window at the native SAR scale.

ratio_glcm_raw = ratio_glcm_uint8.glcmTexture(size=3)

print("\nRaw S1 GLCM bands:")
print(ratio_glcm_raw.bandNames().getInfo())

# Select two interpretable metrics only:
# contrast = local spatial contrast / heterogeneity
# entropy = spatial disorder / complexity

s1_glcm_image = (
    ratio_glcm_raw
    .select(
        ["ratio_contrast", "ratio_ent"],
        ["ratio_glcm_contrast", "ratio_glcm_entropy"]
    )
    .clip(roi)
)

print("\nS1 GLCM bands selected:")
print(s1_glcm_image.bandNames().getInfo())

# ------------------------------------------------------------
# Extract GLCM values by polygon
# ------------------------------------------------------------

s1_glcm_fc = s1_glcm_image.reduceRegions(
    collection=samples_fc,
    reducer=ee.Reducer.mean(),
    scale=10,
    tileScale=4
)

print("\nExtracting S1 GLCM attributes to pandas DataFrame...")

s1_glcm_df = geemap.ee_to_df(s1_glcm_fc)

# ------------------------------------------------------------
# Clean output table
# ------------------------------------------------------------

keep_cols = [
    "sample_id",
    "class",
    "New_ID",
    "shade_level",
    "group",
    "ratio_glcm_contrast",
    "ratio_glcm_entropy"
]

available_cols = [c for c in keep_cols if c in s1_glcm_df.columns]
s1_glcm_df = s1_glcm_df[available_cols].copy()

s1_glcm_df["class"] = s1_glcm_df["class"].astype(str).str.strip().str.lower()
s1_glcm_df["group"] = s1_glcm_df["group"].astype(str).str.strip().str.lower()

s1_glcm_df["New_ID"] = pd.to_numeric(s1_glcm_df["New_ID"], errors="coerce")
s1_glcm_df["shade_level"] = pd.to_numeric(s1_glcm_df["shade_level"], errors="coerce")

s1_glcm_df.loc[s1_glcm_df["New_ID"] == -9999, "New_ID"] = np.nan
s1_glcm_df.loc[s1_glcm_df["shade_level"] == -9999, "shade_level"] = np.nan
s1_glcm_df.loc[s1_glcm_df["class"] == "forest", "New_ID"] = np.nan
s1_glcm_df.loc[s1_glcm_df["class"] == "forest", "shade_level"] = np.nan

for col in ["ratio_glcm_contrast", "ratio_glcm_entropy"]:
    if col in s1_glcm_df.columns:
        s1_glcm_df[col] = pd.to_numeric(s1_glcm_df[col], errors="coerce")

print("\nS1 GLCM table shape:", s1_glcm_df.shape)

print("\nMissing values per S1 GLCM feature:")
print(s1_glcm_df[["ratio_glcm_contrast", "ratio_glcm_entropy"]].isna().sum())

print("\nFirst rows of S1 GLCM table:")
display(s1_glcm_df.head())

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

os.makedirs(TABLE_OUTPUT_DIR, exist_ok=True)

s1_glcm_csv = os.path.join(
    TABLE_OUTPUT_DIR,
    "S1_GLCM_texture_DJFM_3class_optional.csv"
)

s1_glcm_df.to_csv(s1_glcm_csv, index=False)

print("\nS1 GLCM CSV exported to:")
print(s1_glcm_csv)

print("\n" + "="*70)
print("END OF SENTINEL-1 GLCM TEXTURE BLOCK")
print("="*70)


## Block 7C — Optional Sentinel-1 GLCM separability diagnostics

This block evaluates whether the S1 texture variables derived from `DJFM_22_25_ratio_med` show meaningful univariate separability between coffee and forest. Use these diagnostics before deciding whether S1 GLCM should enter the final integration model.


In [ ]:
# ============================================================
# REQUIRED IMPORTS FOR S1 GLCM SEPARABILITY
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu

print("Required imports loaded successfully.")

In [ ]:
# ============================================================
# OPTIONAL BLOCK — S1 GLCM SEPARABILITY DIAGNOSTICS
# Coffee vs forest
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu

print("\n" + "="*70)
print("S1 GLCM SEPARABILITY — COFFEE VS FOREST")
print("="*70)

def cohens_d(x1, x0):
    x1 = pd.Series(x1).dropna().astype(float)
    x0 = pd.Series(x0).dropna().astype(float)

    n1 = len(x1)
    n0 = len(x0)

    if n1 < 2 or n0 < 2:
        return np.nan

    pooled_sd = np.sqrt(
        ((n1 - 1) * x1.var(ddof=1) + (n0 - 1) * x0.var(ddof=1)) /
        (n1 + n0 - 2)
    )

    if pooled_sd == 0:
        return np.nan

    return (x1.mean() - x0.mean()) / pooled_sd


def classify_auc(value):
    if value >= 0.80:
        return "strong"
    elif value >= 0.70:
        return "moderate"
    elif value >= 0.60:
        return "weak"
    else:
        return "very weak"


def classify_effect(value):
    value = abs(value)

    if value >= 0.80:
        return "large"
    elif value >= 0.50:
        return "medium"
    elif value >= 0.20:
        return "small"
    else:
        return "very small"


s1_glcm_features = [
    "ratio_glcm_contrast",
    "ratio_glcm_entropy"
]

s1_glcm_eval_df = s1_glcm_df.copy()
s1_glcm_eval_df["class"] = s1_glcm_eval_df["class"].astype(str).str.lower()

s1_glcm_eval_df = s1_glcm_eval_df[
    s1_glcm_eval_df["class"].isin(["coffee", "forest"])
].copy()

s1_glcm_eval_df["y"] = (s1_glcm_eval_df["class"] == "coffee").astype(int)

records = []

for feature in s1_glcm_features:
    if feature not in s1_glcm_eval_df.columns:
        print(f"[WARNING] Feature not found: {feature}")
        continue

    valid_df = s1_glcm_eval_df[["class", "y", feature]].dropna().copy()

    coffee_values = valid_df.loc[valid_df["class"] == "coffee", feature]
    forest_values = valid_df.loc[valid_df["class"] == "forest", feature]

    if len(coffee_values) < 2 or len(forest_values) < 2:
        print(f"[WARNING] Not enough valid values for {feature}.")
        continue

    auc_raw = roc_auc_score(valid_df["y"], valid_df[feature])
    auc = max(auc_raw, 1 - auc_raw)

    try:
        _, p_value = mannwhitneyu(
            coffee_values,
            forest_values,
            alternative="two-sided"
        )
    except Exception:
        p_value = np.nan

    d_value = cohens_d(coffee_values, forest_values)

    records.append(
        {
            "Feature": feature,
            "AUC": auc,
            "AUC_raw": auc_raw,
            "Cohens_d": d_value,
            "abs_Cohens_d": abs(d_value),
            "p_value": p_value,
            "Mean_coffee": coffee_values.mean(),
            "Mean_forest": forest_values.mean(),
            "Diff_forest_minus_coffee": forest_values.mean() - coffee_values.mean(),
            "n_coffee": len(coffee_values),
            "n_forest": len(forest_values),
            "AUC_class": classify_auc(auc),
            "Effect_class": classify_effect(d_value)
        }
    )

s1_glcm_sep_df = pd.DataFrame(records)

if not s1_glcm_sep_df.empty:
    s1_glcm_sep_df = s1_glcm_sep_df.sort_values(
        ["AUC", "abs_Cohens_d"],
        ascending=False
    ).reset_index(drop=True)

print("\nS1 GLCM separability table:")
display(s1_glcm_sep_df)

print("\nS1 GLCM separability — copy-ready:")
for _, row in s1_glcm_sep_df.iterrows():
    print(
        f"{row['Feature']}: "
        f"AUC = {row['AUC']:.4f}; "
        f"Cohen_d = {row['Cohens_d']:.4f}; "
        f"Mean_coffee = {row['Mean_coffee']:.6f}; "
        f"Mean_forest = {row['Mean_forest']:.6f}; "
        f"p_value = {row['p_value']:.3e}; "
        f"AUC_class = {row['AUC_class']}; "
        f"Effect_class = {row['Effect_class']}"
    )

print("\n" + "="*70)
print("END OF S1 GLCM SEPARABILITY")
print("="*70)

## Block 8 — SAR separability analysis

This block evaluates the univariate separability of the SAR features between coffee and forest, and among coffee shade groups.

The analysis uses AUC, Cohen's d and non-parametric tests as descriptive diagnostics. These statistics are not intended to replace the multivariate model, but they help interpret which SAR variables contain useful class information.


In [ ]:
from scipy.stats import mannwhitneyu, kruskal
from sklearn.metrics import roc_auc_score
from itertools import combinations

# Main SAR analytical features
sar_main_features = [
    "DJFM_22_25_VV_med",
    "DJFM_22_25_VH_med",
    "DJFM_22_25_ratio_med",
    "DJFM_22_25_ratio_sd",
    "DJFM_22_25_ratio_p90_minus_p10"
]

# QA fields retained only for diagnostics
sar_qa_features = [
    "DJFM_22_25_VV_count",
    "DJFM_22_25_angle_med"
]

sar_main_features = [f for f in sar_main_features if f in sar_df.columns]
sar_qa_features = [f for f in sar_qa_features if f in sar_df.columns]

print("Main SAR features:")
print(sar_main_features)
print("\nQA SAR fields:")
print(sar_qa_features)

required_cols = ["class", "sample_id", "New_ID", "shade_level", "group"] + sar_main_features
missing = [c for c in required_cols if c not in sar_df.columns]
if missing:
    raise ValueError(f"Missing columns in sar_df: {missing}")

coffee_sar = sar_df[sar_df["class"] == "coffee"].copy()
forest_sar = sar_df[sar_df["class"] == "forest"].copy()

group_order = [
    "forest",
    "coffee_low_shade",
    "coffee_intermediate_shade",
    "coffee_high_shade"
]

group_labels = {
    "forest": "Forest",
    "coffee_low_shade": "Coffee - low shade",
    "coffee_intermediate_shade": "Coffee - intermediate shade",
    "coffee_high_shade": "Coffee - high shade"
}

sar_groups_df = sar_df[sar_df["group"].isin(group_order)].copy()

def cohens_d(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    x = x[~np.isnan(x)]
    y = y[~np.isnan(y)]

    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan

    pooled_std = np.sqrt(
        ((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1))
        / (nx + ny - 2)
    )

    if pooled_std == 0:
        return np.nan

    return (np.mean(x) - np.mean(y)) / pooled_std

def classify_auc(auc):
    if auc >= 0.90:
        return "very strong"
    if auc >= 0.80:
        return "strong"
    if auc >= 0.70:
        return "moderate"
    if auc >= 0.60:
        return "weak"
    return "very weak"

def classify_d(d):
    ad = abs(d)
    if ad >= 1.20:
        return "very large"
    if ad >= 0.80:
        return "large"
    if ad >= 0.50:
        return "medium"
    if ad >= 0.20:
        return "small"
    return "very small"

def classify_p(p):
    if p < 0.01:
        return "strong evidence"
    if p < 0.05:
        return "evidence"
    return "insufficient"

In [ ]:
# Binary SAR separability: coffee vs forest
sar_rows = []

for feat in sar_main_features:
    x = coffee_sar[feat].dropna()
    y = forest_sar[feat].dropna()

    if len(x) < 2 or len(y) < 2:
        continue

    y_true = np.concatenate([np.ones(len(x)), np.zeros(len(y))])
    y_scores = np.concatenate([x.values, y.values])

    auc_raw = roc_auc_score(y_true, y_scores)
    auc = max(auc_raw, 1 - auc_raw)
    d = cohens_d(x, y)
    p = mannwhitneyu(x, y, alternative="two-sided").pvalue

    sar_rows.append({
        "Feature": feat,
        "AUC": auc,
        "AUC_raw": auc_raw,
        "Cohens_d": d,
        "abs_Cohens_d": abs(d),
        "p_value": p,
        "Mean_coffee": x.mean(),
        "Mean_forest": y.mean(),
        "Diff_forest_minus_coffee": y.mean() - x.mean(),
        "n_coffee": len(x),
        "n_forest": len(y)
    })

sar_sep_df = pd.DataFrame(sar_rows).sort_values("AUC", ascending=False).reset_index(drop=True)
sar_sep_df["AUC_class"] = sar_sep_df["AUC"].apply(classify_auc)
sar_sep_df["Effect_class"] = sar_sep_df["Cohens_d"].apply(classify_d)
sar_sep_df["P_class"] = sar_sep_df["p_value"].apply(classify_p)

display(sar_sep_df)

In [ ]:
# Pairwise separability among forest and coffee shade groups
pairwise_sar_rows = []

for g1, g2 in combinations(group_order, 2):
    df1 = sar_groups_df[sar_groups_df["group"] == g1]
    df2 = sar_groups_df[sar_groups_df["group"] == g2]

    for feat in sar_main_features:
        x = df1[feat].dropna()
        y = df2[feat].dropna()

        if len(x) < 2 or len(y) < 2:
            continue

        y_true = np.concatenate([np.ones(len(x)), np.zeros(len(y))])
        y_scores = np.concatenate([x.values, y.values])

        auc_raw = roc_auc_score(y_true, y_scores)
        auc = max(auc_raw, 1 - auc_raw)
        d = cohens_d(x, y)
        p = mannwhitneyu(x, y, alternative="two-sided").pvalue

        pairwise_sar_rows.append({
            "group_1": g1,
            "group_2": g2,
            "comparison": f"{group_labels[g1]} vs {group_labels[g2]}",
            "Feature": feat,
            "AUC": auc,
            "AUC_raw": auc_raw,
            "Cohens_d": d,
            "abs_Cohens_d": abs(d),
            "p_value": p,
            "Mean_group_1": x.mean(),
            "Mean_group_2": y.mean(),
            "Diff_group_1_minus_group_2": x.mean() - y.mean(),
            "n_group_1": len(x),
            "n_group_2": len(y)
        })

pairwise_sar_df = pd.DataFrame(pairwise_sar_rows)
pairwise_sar_df["AUC_class"] = pairwise_sar_df["AUC"].apply(classify_auc)
pairwise_sar_df["Effect_class"] = pairwise_sar_df["Cohens_d"].apply(classify_d)
pairwise_sar_df["P_class"] = pairwise_sar_df["p_value"].apply(classify_p)

pairwise_sar_df = pairwise_sar_df.sort_values(
    ["comparison", "AUC"],
    ascending=[True, False]
).reset_index(drop=True)

display(pairwise_sar_df)

## Block 9 — Random Forest SAR-only model

This block trains and evaluates the SAR-only Random Forest scenario using stratified five-fold cross-validation and out-of-fold predictions.

The objective is not to optimise a standalone operational classifier, but to quantify the detectability provided by Sentinel-1 SAR metrics alone and compare it with the optical and fusion scenarios reported in the manuscript.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)

df_model_sar = sar_df.copy()

# Main SAR predictors only: no count, no angle
sar_model_features = sar_main_features.copy()

required_cols = ["class", "sample_id", "New_ID", "shade_level", "group"] + sar_model_features
missing = [c for c in required_cols if c not in df_model_sar.columns]
if missing:
    raise ValueError(f"Missing columns in df_model_sar: {missing}")

df_model_sar = df_model_sar[df_model_sar["class"].isin(["coffee", "forest"])].copy()
df_model_sar = df_model_sar.dropna(subset=sar_model_features).copy()

df_model_sar["target"] = df_model_sar["class"].map({"forest": 0, "coffee": 1})

X_sar = df_model_sar[sar_model_features]
y_sar = df_model_sar["target"]

rf_sar = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

y_pred_sar = cross_val_predict(
    rf_sar,
    X_sar,
    y_sar,
    cv=cv,
    method="predict",
    n_jobs=-1
)

y_prob_sar = cross_val_predict(
    rf_sar,
    X_sar,
    y_sar,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

sar_cv_results = df_model_sar.copy()
sar_cv_results["y_true"] = y_sar.values
sar_cv_results["y_pred"] = y_pred_sar
sar_cv_results["y_prob_coffee"] = y_prob_sar
sar_cv_results["correct"] = sar_cv_results["y_true"] == sar_cv_results["y_pred"]
sar_cv_results["predicted_class"] = sar_cv_results["y_pred"].map({0: "forest", 1: "coffee"})

print("SAR model dataset shape:", df_model_sar.shape)
print("\nClass distribution:")
print(df_model_sar["class"].value_counts())
print("\nSAR features used:")
print(sar_model_features)

In [ ]:
# Overall SAR performance
sar_acc = accuracy_score(y_sar, y_pred_sar)
sar_prec = precision_score(y_sar, y_pred_sar)
sar_rec = recall_score(y_sar, y_pred_sar)
sar_f1 = f1_score(y_sar, y_pred_sar)
sar_auc = roc_auc_score(y_sar, y_prob_sar)
sar_ap = average_precision_score(y_sar, y_prob_sar)

sar_metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC AUC",
        "Average precision"
    ],
    "Value": [
        sar_acc,
        sar_prec,
        sar_rec,
        sar_f1,
        sar_auc,
        sar_ap
    ]
})

display(sar_metrics_df)

print("Classification report:")
print(
    classification_report(
        y_sar,
        y_pred_sar,
        target_names=["Forest", "Coffee"],
        digits=4
    )
)

cm_sar = confusion_matrix(y_sar, y_pred_sar)
cm_sar_df = pd.DataFrame(
    cm_sar,
    index=["True Forest", "True Coffee"],
    columns=["Predicted Forest", "Predicted Coffee"]
)

display(cm_sar_df)

In [ ]:
# Final RF feature importance, trained once on the full SAR dataset
rf_sar_final = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_sar_final.fit(X_sar, y_sar)

sar_importance_df = pd.DataFrame({
    "Feature": sar_model_features,
    "Importance": rf_sar_final.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

display(sar_importance_df)

## Block 10 — Error analysis

This block identifies the polygons misclassified by the SAR-only model and summarises coffee omission by shade group.

These diagnostics support the interpretation of detectability limits, especially whether errors are associated with increasing shade level or with broader spectral-structural convergence between tree-covered coffee and forest.


In [ ]:
# Individual SAR errors
sar_errors = sar_cv_results[sar_cv_results["correct"] == False].copy()

sar_errors = sar_errors[
    [
        "sample_id",
        "class",
        "shade_level",
        "group",
        "y_true",
        "y_pred",
        "predicted_class",
        "y_prob_coffee"
    ] + sar_model_features
].sort_values(["class", "group", "y_prob_coffee"])

display(sar_errors)

print("SAR errors by true class:")
print(sar_errors["class"].value_counts())

print("\nSAR errors by group:")
print(sar_errors["group"].value_counts())

In [ ]:
# Coffee omission by shade level
coffee_sar_cv = sar_cv_results[sar_cv_results["class"] == "coffee"].copy()

shade_error_sar_summary = (
    coffee_sar_cv
    .groupby("group")
    .agg(
        n=("sample_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        mean_prob_coffee=("y_prob_coffee", "mean"),
        sd_prob_coffee=("y_prob_coffee", "std"),
        min_prob_coffee=("y_prob_coffee", "min"),
        max_prob_coffee=("y_prob_coffee", "max")
    )
    .reset_index()
)

shade_error_sar_summary["misclassified"] = (
    shade_error_sar_summary["n"] - shade_error_sar_summary["correct"]
)

shade_error_sar_summary["error_rate"] = (
    shade_error_sar_summary["misclassified"] / shade_error_sar_summary["n"]
)

shade_order = [
    "coffee_low_shade",
    "coffee_intermediate_shade",
    "coffee_high_shade"
]

shade_error_sar_summary["group"] = pd.Categorical(
    shade_error_sar_summary["group"],
    categories=shade_order,
    ordered=True
)

shade_error_sar_summary = shade_error_sar_summary.sort_values("group")
display(shade_error_sar_summary)

## Block 11 — Optional figure export helper

This block provides a simple helper function for exporting figures in high-resolution TIFF format. Figure export is optional and depends on the needs of the manuscript, supplementary material or repository documentation.


In [ ]:
os.makedirs(FIGURE_OUTPUT_DIR, exist_ok=True)

def save_tiff(fig, filename, dpi=300):
    out_path = os.path.join(FIGURE_OUTPUT_DIR, filename)
    fig.savefig(
        out_path,
        dpi=dpi,
        format="tiff",
        bbox_inches="tight",
    )
    print(f"Figure saved to: {out_path}")


In [ ]:
# Example figure: SAR feature importance
plot_df = sar_importance_df.copy()

label_map = {
    "DJFM_22_25_VV_med": "VV backscatter",
    "DJFM_22_25_VH_med": "VH backscatter",
    "DJFM_22_25_ratio_med": "VH/VV ratio",
    "DJFM_22_25_ratio_sd": "VH/VV ratio SD",
    "DJFM_22_25_ratio_p90_minus_p10": "VH/VV ratio p90–p10"
}

plot_df["Label"] = plot_df["Feature"].map(label_map).fillna(plot_df["Feature"])

fig, ax = plt.subplots(figsize=(8, 4.8))

ax.barh(
    plot_df["Label"][::-1],
    plot_df["Importance"][::-1]
)

ax.set_xlabel("Random Forest importance")
ax.set_ylabel("")
ax.set_title("Feature importance — SAR Random Forest")
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
save_tiff(fig, "SAR_RF_feature_importance_main_noDEM.tiff", dpi=300)
plt.show()

## Block 12 — Reproducibility summary for reporting

This block prints a compact summary of the SAR-only scenario, including performance metrics, confusion matrix values and feature importance.

The output is intended to support manuscript reporting, supplementary documentation and consistency checks across the other notebooks in the repository.


In [ ]:
# ============================================================
# REPRODUCIBILITY SUMMARY — SENTINEL-1 SAR
# ============================================================

print("\n" + "="*80)
print("SENTINEL-1 SAR REPRODUCIBILITY SUMMARY")
print("="*80)

print("\nSAR MODEL:")
try:
    metric_map = dict(zip(sar_metrics_df["Metric"], sar_metrics_df["Value"]))
    print(f"Accuracy = {metric_map.get('Accuracy', np.nan):.4f}")
    print(f"Precision = {metric_map.get('Precision', np.nan):.4f}")
    print(f"Recall = {metric_map.get('Recall', np.nan):.4f}")
    print(f"F1-score = {metric_map.get('F1-score', np.nan):.4f}")
    print(f"ROC AUC = {metric_map.get('ROC AUC', np.nan):.4f}")
    print(f"Average precision = {metric_map.get('Average precision', np.nan):.4f}")
except Exception as e:
    print("[ERROR] Could not print SAR metrics.")
    print(e)

print("\nSAR CONFUSION MATRIX:")
try:
    print(f"True Forest / Predicted Forest = {cm_sar_df.loc['True Forest', 'Predicted Forest']}")
    print(f"True Forest / Predicted Coffee = {cm_sar_df.loc['True Forest', 'Predicted Coffee']}")
    print(f"True Coffee / Predicted Forest = {cm_sar_df.loc['True Coffee', 'Predicted Forest']}")
    print(f"True Coffee / Predicted Coffee = {cm_sar_df.loc['True Coffee', 'Predicted Coffee']}")
except Exception as e:
    print("[ERROR] Could not print SAR confusion matrix.")
    print(e)

print("\nSAR FEATURE IMPORTANCE:")
try:
    for _, row in sar_importance_df.iterrows():
        print(f"{row['Feature']} = {row['Importance']:.6f}")
except Exception as e:
    print("[ERROR] Could not print SAR feature importance.")
    print(e)

print("\nSAR FEATURES USED IN THE MODEL:")
try:
    for f in sar_model_features:
        print(f"- {f}")
except Exception as e:
    print("[ERROR] Could not print SAR feature list.")
    print(e)

print("\nS1 GLCM SEPARABILITY — IF AVAILABLE:")
if "s1_glcm_sep_df" in globals() and isinstance(s1_glcm_sep_df, pd.DataFrame) and not s1_glcm_sep_df.empty:
    for _, row in s1_glcm_sep_df.iterrows():
        print(
            f"{row['Feature']}: "
            f"AUC = {row['AUC']:.4f}; "
            f"Cohen_d = {row['Cohens_d']:.4f}; "
            f"Mean_coffee = {row['Mean_coffee']:.6f}; "
            f"Mean_forest = {row['Mean_forest']:.6f}; "
            f"p_value = {row['p_value']:.3e}; "
            f"AUC_class = {row['AUC_class']}; "
            f"Effect_class = {row['Effect_class']}"
        )
else:
    print("S1 GLCM separability was not computed or produced no valid results.")

print("\n" + "="*80)
print("END OF SENTINEL-1 SAR REPRODUCIBILITY SUMMARY")
print("="*80)


## Recommended data-sharing approach for this repository

For this study, the safest public-repository strategy is to share code and documentation while keeping the original georeferenced field polygons outside GitHub.

Recommended public contents:

- the notebooks required to reproduce the workflow;
- a `data/README.md` describing the required input schema;
- optional anonymised or derived CSV tables without coordinates, producer names or property identifiers;
- optional example data with synthetic geometries or spatially degraded locations.

Avoid committing:

- raw shapefiles or GeoPackages with exact field locations;
- producer names, farm identifiers or private institutional attributes;
- Google Earth Engine credentials, service-account keys or cloud project identifiers;
- large raster exports such as GeoTIFFs unless they are explicitly intended for public release.


## Statement on the use of large language models

Large language models were used to assist with notebook organisation, code documentation and language refinement. All scientific decisions, data-processing choices, modelling procedures, results and interpretations were reviewed, verified and approved by the author(s). No confidential data, private field geometries or restricted institutional datasets should be included in the public repository.
